# Export and Import Document schema from a processor using Gemini 

* Author: docai-incubator@google.com

## Disclaimer

This tool is not supported by the Google engineering team or product team. It is provided and supported on a best-effort basis by the DocAI Incubator Team. No guarantees of performance are implied.

## Objective

This document guides how to extract and export a schema from a sample document to a spreadsheet (.xlsx extension) using Gemini and import a schema from a spreadsheet to a processor.

## Prerequisites

* Agent Platform Notebook Or Colab (If using Colab, use authentication)
* Storage Bucket for storing input and output json files
* Permission For Google Storage and Agent Platform Notebook.
* Agent Platform API enabled for Gemini API calls
* Processor details to import to the processor


## Authentication (needed if you run the code in colab)

In [ ]:
from google.colab import auth
from google.auth import default

cred = auth.authenticate_user()
creds, _ = default()

## 1. Generating DocumentAI schema in a excel format using GEMINI model and Document

### Input and Output Paths


In [ ]:
#Input
project_id = "xxxx-xxxx-xxx" # Project ID of the project
location = "us-central1" # Location of Gemini
mime_type ="application/pdf" # Mime type of input document
input_uri = "gs://xxxx/xxxx/xxx/xxx.pdf" # GCS uri of input document
output_gcs_bucket = "xxxxxx"
output_gcs_folder = "xxxxx/xxxxxx"

* ``project_id`` : It should contains the project id of your current project.
* ``location ``: location of Gemini Model.
* ``mime_type ``: Mime type of document in gcs storage bucket which will be given as sample to generate schema
* ``input_uri`` : GCS URI of document which will be given as sample to generate schema
* ``output_gcs_bucket`` : GCS bucket to store the schema excel file generated
* ``output_gcs_folder``:  GCS folder path excluding bucket name to store the schema excel file generated

### Run the below code to generate schema using Gemini Model

In [ ]:
import os
import json
from typing import Any, Dict, List, Union, Literal
import pandas as pd
from google import genai
from google.genai import types
from google.cloud import storage
from pydantic import BaseModel, Field

# ==========================================
# 1. DEFINE STRUTURED OUTPUT SCHEMA (PYDANTIC)
# ==========================================
class SchemaItem(BaseModel):
    name: str = Field(
        ..., 
        description="The title of the field. This should be a semantically named field, less than 64 characters and snake_cased. Use underscores to join multiple words."
    )
    value_type: str = Field(
        ..., 
        description="Type of field: 'string', 'number', 'currency', 'money', 'datetime', 'address', or 'checkbox'. If the field is a parent entity (has child fields nested inside), value_type should equal the 'name'."
    )
    occurrence_type: Literal["REQUIRED_ONCE", "REQUIRED_MULTIPLE", "OPTIONAL_ONCE", "OPTIONAL_MULTIPLE"] = Field(
        ..., 
        description="Expected number of times the field appears. Use REQUIRED_* for identifiers or mandatory fields, and OPTIONAL_* for most other cases."
    )
    display_name: str = Field(
        "", 
        description="Leave blank unless the entity is a child entity, in which case display_name must be equal to the parent entity's name."
    )
    description: str = Field(
        ..., 
        description="A prompt under 500 characters defining the entity. This guides Document AI's extraction model."
    )

# ==========================================
# 2. DEFINE SYSTEM INSTRUCTION PROMPT & CONFIGS
# ==========================================
text1 = """Please analyze the structure of the attached document to define its schema. 
This schema will be used to configure and train a custom Document AI Processor.

The schema definition must include:
* Fields and their data types (e.g., text, number, date)
* Any relationships between fields (e.g., nested objects, arrays)
* Each object in the schema must have the key names: name, value_type, occurrence_type, display_name, and description.

Please follow these strict attribute-level guidelines:
* All 'name' keys should be less than 64 characters, semantically named, and written in snake_case (replace spaces with underscores).
* 'value_type' describes the field's data type. It must be one of: 'string', 'number', 'currency', 'money', 'datetime', 'address', or 'checkbox'. If the field is a parent entity (has child fields nested inside), the 'value_type' should be equal to the 'name'.
* 'occurrence_type' describes how often the field appears and must be either: 'REQUIRED_ONCE', 'REQUIRED_MULTIPLE', 'OPTIONAL_ONCE', or 'OPTIONAL_MULTIPLE'. Use REQUIRED_* for identifiers and critical fields.
* 'display_name' must remain blank unless the entity is a child entity. If it is a child, the 'display_name' must equal the parent entity's 'name'.
* 'description' must be a prompt under 500 characters that defines the entity. Write this prompt to help guide the Document AI extraction model to find and extract the correct data from documents.

Ensure that the schema is completely aligned with the Document AI custom processor schema format. Retain the exact language of label names as found in the original document."""

# Configured safety settings using the google-genai SDK format
safety_settings = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=types.HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    ),
]

# ==========================================
# 3. SCHEMA GENERATION FUNCTION
# ==========================================
def generate(project_id : str, location : str, mime_type : str, input_uri : str) -> List[Dict[str, Any]]:
    """Generates a structured JSON schema from a multi-modal document using Gemini.

    Args:
        project_id: The Google Cloud Current Project ID.
        location: The region for the Gemini Model (e.g., 'us-central1').
        mime_type: The MIME type of the input document (e.g., 'application/pdf').
        input_uri: GCS URI of document which will be given as sample to generate schema

    Returns:
        A list of dictionaries representing the parsed JSON schema generated by the model.
    """
    # INITIALIZED WITH vertexai=True (Matches Python SDK spec)
    client = genai.Client(vertexai=True, project=project_id, location=location)
    
    print(f"Generating schema using the document {input_uri}...")
    
    # Instantiate the Part payload safely inside the function (fixes scoping bugs)
    document1 = types.Part.from_uri(
        mime_type=mime_type,
        file_uri=input_uri,
    )
    
    # Configure generation options to enforce output using your Pydantic schema structure
    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[SchemaItem],
        temperature=1.0,
        max_output_tokens=8192,
        top_p=0.95,
        safety_settings=safety_settings,
    )
    
    # Call upgraded model (gemini-3.5-flash) using prompt + document
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[text1, document1],
        config=config,
    )
    
    # Safely convert Pydantic parsed results directly to standard dicts for pandas Excel export
    schema_json = [item.model_dump() for item in response.parsed]
    return schema_json

# ==========================================
# 4. GCS UPLOAD & LOCAL CLEANUP FUNCTIONS
# ==========================================
def upload_to_gcs(bucket_name : str, destination_blob_name : str, source_file_path : str) -> None:
    """Uploads a local file to a specified Google Cloud Storage bucket.

    Args:
        bucket_name: The name of the GCS bucket (e.g., 'my-bucket-name').
        destination_blob_name: The destination path and file name inside the bucket.
        source_file_path: The local path to the file that needs to be uploaded.
    """
    # Initialize standard storage client
    storage_client = storage.Client()
    # Get the target bucket
    bucket = storage_client.bucket(bucket_name)
    # Create a new blob (file) in the bucket
    blob = bucket.blob(destination_blob_name)
    # Upload the file to the blob
    blob.upload_from_filename(source_file_path)
    print(f"File {source_file_path} uploaded to {destination_blob_name}")

def remove_local_file(file_path : str) -> None:
    """Removes a specified file from the local file system with safety exception handling."""
    try:
        os.remove(file_path)
        print(f"File {file_path} has been removed from local storage.")
    except FileNotFoundError:
        print(f"File {file_path} not found.")
    except PermissionError:
        print(f"Permission denied: Unable to delete {file_path}.")
    except Exception as e:
        print(f"Error occurred while trying to delete {file_path}: {e}")

# ==========================================
# 5. SCHEMA EXPORT FUNCTION (BACKWARD COMPATIBLE)
# ==========================================
def export_schema(document_schema : Union[List[Dict[str, Any]], Dict[str, Any]],
                  output_bucket : str,
                  output_folder : str,
                  source_filename : str) -> None:
    """Converts a schema payload to an Excel file, saves it locally, and uploads it to GCS."""
    schema_filename = f"{source_filename}_schema_exported.xlsx"
    df = pd.DataFrame(document_schema)
    df.to_excel(schema_filename, index=False)
    
    try:
        upload_to_gcs(
            output_bucket,
            f"{output_folder.rstrip('/')}/{schema_filename}",
            schema_filename
        )
    except Exception as e:
        print("Error occurred while uploading schema to bucket")
        print(e)
        
    remove_local_file(schema_filename)


gen_schema_json = generate(project_id, location, mime_type, input_uri)

export_schema(
    gen_schema_json, 
    output_gcs_bucket, 
    output_gcs_folder,
    input_uri.split("/")[-1].split(".") 
)

## 2. Import schema from Excel file to processor


### Input

In [ ]:
project_id="rand-automl-project" # Project ID of the project
new_location="us" # location of the processor 
new_processor_id="69fb7e829e443bff" #Processor id of processor to which the schema has to be imported
schema_bucket_name = "test_vb1" # Bucket name where exported schema file is stored
schema_file_path = "Gemini_schema_test/output/['bank_stmt_test_original_docs_10119_0-26591_0-hgf_4885721002541008_bank_statement_blank_a (1)', 'pdf']_schema_exported.xlsx"

* ``project_id`` : It should contains the project id of your current project.
* ``new_location ``: location of new documentai processor.
* ``new_processor_id ``: Processor id to which the schema has to be imported
* ``input_uri`` : GCS URI of document which will be given as sample to generate schema
* ``schema_bucket_name`` : GCS bucket where schema excel exists
* ``schema_file_path``:  GCS folder path excluding bucket name where schema excel exists

### Importing libraries

In [ ]:
from collections import defaultdict
import math
import pandas as pd
from google.cloud import documentai_v1beta3 as documentai
from typing import Any, Dict, List, Union

from google.cloud import storage
from typing import Optional
from google.api_core.client_options import ClientOptions
from google.api_core.exceptions import GoogleAPICallError

### Reading excel file

In [ ]:
def download_from_gcs(bucket_name : str, source_blob_name : str, destination_file_path : str) -> None:
    """Downloads a file from a specified Google Cloud Storage bucket to local storage.

    Args:
        bucket_name: The name of the GCS bucket (e.g., 'my-bucket-name').
        source_blob_name: The path and name of the file inside the 
            bucket (e.g., 'folder/document.pdf').
        destination_file_path: The local path where the file should be 
            saved (e.g., '/local/path/downloads/document.pdf').

    Returns:
        None
    """
    # Initialize a client
    storage_client = storage.Client()

    # Get the bucket
    bucket = storage_client.bucket(bucket_name)

    # Get the blob (file) in the bucket
    blob = bucket.blob(source_blob_name)

    # Download the file from the bucket to the local storage
    blob.download_to_filename(destination_file_path)

    print(f"File {source_blob_name} downloaded to {destination_file_path}.")

local_excel_path = schema_file_path.split("/")[-1]

download_from_gcs(schema_bucket_name, schema_file_path, local_excel_path)

# time.sleep(10)

# Import the Excel file back into a data frame
imported_df = pd.read_excel(local_excel_path, engine="openpyxl")

# Convert the data frame back to a list of dictionaries
imported_data = imported_df.to_dict(orient='records')


### Required functions

In [ ]:
def group_by_display_name(dict_list : List[Dict[str, Any]]) -> Dict[str, List[Dict[str, Any]]]:
    """Groups a list of dictionaries by their 'display_name' key.

    Args:
        dict_list: A list of dictionaries, where each dictionary represents 
            an item containing metadata (optionally including a 'display_name').

    Returns:
        A dictionary where keys are unique 'display_name' strings and values 
        are lists of dictionaries belonging to that group.
    """
    grouped = defaultdict(list)

    for item in dict_list:
        key = item.get("display_name", "")
        grouped[key].append(item)

    return dict(grouped)

def create_entity_schema(temp_dict : Dict[str, Any]) -> documentai.DocumentSchema.EntityType.Property:
    """Creates a Document AI entity schema property, handling potential NaN values."""
    name = temp_dict.get('name')
    display_name = temp_dict.get('display_name')
    description = temp_dict.get('description')
    return documentai.DocumentSchema.EntityType.Property(
        name=name if not (isinstance(name, float) and math.isnan(name)) else '',
        display_name=display_name if not (isinstance(display_name, float) and math.isnan(display_name)) else '',
        value_type=temp_dict.get('value_type', ''),
        occurrence_type=temp_dict.get('occurrence_type', 'UNSPECIFIED'),
        description=description if not (isinstance(description, float) and math.isnan(description)) else '',
    )


### RUN THE BELOW CODE TO EXPORT SCHEMA TO PROCESSOR
###    (Note: Any schema existing in the processor will be overrided)

In [ ]:
### 1. Initialize Client and Define Names ###
opts = ClientOptions(api_endpoint=f"{new_location}-documentai.googleapis.com")
client = documentai.DocumentServiceClient(client_options=opts)
processor_name = f'projects/{project_id}/locations/{new_location}/processors/{new_processor_id}'
dataset_schema_name = f"{processor_name}/dataset/datasetSchema"

### 2. Separate Root Properties from Nested Properties ###
root_properties = []
nested_groups = defaultdict(list)
for item in imported_data:
    display_name = item.get("display_name")
    if display_name is None or (isinstance(display_name, float) and math.isnan(display_name)):
        # This is a root-level property.
        root_properties.append(create_entity_schema(item))
    else:
        # This property belongs inside a nested entity.
        nested_groups[display_name].append(create_entity_schema(item))

### 3. Build the Schema Definitions ###
# This list will hold the definitions for our nested types (e.g., line_items).
nested_entity_types = []

for display_name, properties_list in nested_groups.items():
    # The internal API name for the entity type (e.g., 'line_items').
    entity_type_name = display_name.replace(" ", "_").lower()

    # Define the complex EntityType for the nested group.
    nested_entity = documentai.DocumentSchema.EntityType(
        name=entity_type_name,
        display_name=display_name,
        base_types=["entity"], # Nested, repeatable objects have a base type of 'entity'.
        properties=properties_list
    )
    nested_entity_types.append(nested_entity)

# Create the single root entity. Its properties are ONLY the ones with no display_name.
# We are now assuming this list already contains the property that refers to 'line_items'.
final_root_entity = documentai.DocumentSchema.EntityType(
    name="custom_extraction_document_type",
    display_name="Custom Document Root",
    description="The root entity for the entire document, containing all fields.",
    base_types=["document"], # The single root must have a base type of 'document'.
    properties=root_properties
)

# Construct the final schema object. It must contain the root type AND all nested types.
new_schema = documentai.DocumentSchema(
    display_name="schema_added_dynamically",
    description="Schema with a single root and dynamically generated nested types.",
    entity_types=[final_root_entity] + nested_entity_types
)

### 4. Create and Send the Update Request ###
request = documentai.UpdateDatasetSchemaRequest(
    dataset_schema=documentai.DatasetSchema(
        name=dataset_schema_name,
        document_schema=new_schema
    ),
    update_mask="documentSchema"
)

try:
    print("Sending request to update dataset schema...")
    operation = client.update_dataset_schema(request=request)
    print("\nSchema update operation initiated successfully!")
    print("Check the Google Cloud Console for the processor to see the updated schema.")
except GoogleAPICallError as e:
    print("\n--- API ERROR ---")
    print(f"An error occurred during the schema update: {e}")
except Exception as e:
    print("\n--- UNEXPECTED ERROR ---")
    print(f"An unexpected error occurred: {e}")
